# Lesson 05：GroupBy、Pivot 與 KPI 表

學習目標：
- 使用 `groupby` 依日期彙總營收
- 計算訂單數、總營收與平均訂單金額 AOV
- 用 `pivot_table` 快速建立付款方式報表
- 建立週報並理解時間欄位分組


## 流程大綱

1. 載入資料與完成訂單事實表
2. 每日營收與付款方式 pivot table
3. 每日 KPI 報表
4. 付款方式績效表
5. 週報 KPI
6. 完成一個小練習


## 1. 載入資料

本課沿用 `common.py` 中的三個工具：

- `ensure_packages()`：確認必要套件存在
- `load_data()`：載入 CSV 資料
- `order_facts()`：建立完成訂單層級的事實表，並計算 `line_revenue`


In [2]:
from common import ensure_packages, load_data, order_facts
import pandas as pd

ensure_packages()
data = load_data()
facts = order_facts(data).copy()

print(f"completed order rows: {len(facts):,}")
facts.to_csv("facts.csv")
facts.head()


completed order rows: 20,413


,order_id,line_revenue,customer_id,order_date,status,payment_type
0,1,538.65,2323,2025-05-02,completed,wallet
1,2,4357.70,117,2024-07-14,completed,wallet
3,4,1193.40,1240,2024-07-08,completed,atm
4,5,13175.50,714,2024-08-12,completed,atm
5,6,3657.50,2005,2025-01-14,completed,card


## 2. 每日營收

使用 `facts["order_date"].dt.date` 取出日期，再依日期加總 `line_revenue`。


In [3]:
daily = facts.groupby(facts["order_date"].dt.date, as_index=False)["line_revenue"].sum()

daily.head()


,order_date,line_revenue
0,2024-01-01,172521.55
1,2024-01-02,125992.90
2,2024-01-03,190549.30
3,2024-01-04,176488.30
4,2024-01-05,86529.65


## 3. 付款方式 Pivot Table

`pivot_table` 可以快速建立彙總表。這裡以 `payment_type` 當列，對 `line_revenue` 同時計算筆數、平均值與總和。


In [ ]:
by_payment_pivot = facts.pivot_table(
    values="line_revenue",
    index="payment_type",
    aggfunc=["count", "mean", "sum"],
).round(2)
by_payment_pivot

,count,mean,sum
,line_revenue,line_revenue,line_revenue
payment_type,,,
atm,5111,4979.96,25452594.55
card,10166,4974.20,50567692.45
cod,2060,5052.66,10408480.55
wallet,3076,4987.98,15343032.20


## 4. 每日 KPI 報表

把每日營收擴充成 KPI 表：

- `revenue`：每日總營收
- `orders`：每日訂單數
- `aov`：平均訂單金額，公式是 `revenue / orders`


In [5]:
daily_rev = facts.groupby(facts["order_date"].dt.date, as_index=False).agg(
    revenue=("line_revenue", "sum"),
    orders=("order_id", "count"),
)
daily_rev["aov"] = (daily_rev["revenue"] / daily_rev["orders"]).round(2)

daily_rev.head()


,order_date,revenue,orders,aov
0,2024-01-01,172521.55,34,5074.16
1,2024-01-02,125992.90,32,3937.28
2,2024-01-03,190549.30,36,5293.04
3,2024-01-04,176488.30,26,6788.01
4,2024-01-05,86529.65,19,4554.19


## 5. 練習版：付款方式績效表

這段改用 `groupby().agg()`，欄位命名更清楚，也方便排序。通常營運報表會依總營收或訂單數排序，讓重要類別排在前面。


In [6]:
by_payment = (
    facts.groupby("payment_type", as_index=False)
    .agg(
        orders=("order_id", "count"),
        total_revenue=("line_revenue", "sum"),
        avg_order_value=("line_revenue", "mean"),
    )
    .sort_values("total_revenue", ascending=False)
    .round(2)
)

by_payment


,payment_type,orders,total_revenue,avg_order_value
1,card,10166,50567692.45,4974.20
0,atm,5111,25452594.55,4979.96
3,wallet,3076,15343032.20,4987.98
2,cod,2060,10408480.55,5052.66


## 6. 練習版：週報 KPI

週報的做法是先把 `order_date` 轉成週期欄位 `order_week`，再依週彙總。`dt.to_period("W")` 會把日期轉成週區間。


In [7]:
facts["order_week"] = facts["order_date"].dt.to_period("W")

weekly = facts.groupby("order_week", as_index=False).agg(
    revenue=("line_revenue", "sum"),
    orders=("order_id", "count"),
)
weekly["aov"] = (weekly["revenue"] / weekly["orders"]).round(2)

weekly.head()


,order_week,revenue,orders,aov
0,2024-01-01/2024-01-07,1034534.75,203,5096.23
1,2024-01-08/2024-01-14,964225.40,196,4919.52
2,2024-01-15/2024-01-21,983294.30,200,4916.47
3,2024-01-22/2024-01-28,966631.75,188,5141.66
4,2024-01-29/2024-02-04,1061507.10,213,4983.60


## 7. 解讀重點

看 KPI 表時，可以依序問：

- 營收最高的日期或週期是哪一段？
- 付款方式的營收差異，來自訂單數還是 AOV？
- 如果某週營收變高，是因為訂單變多，還是平均訂單金額變高？

這些問題能把單純的彙總表轉成營運洞察。


## 8. 小練習

請建立一張「每月 KPI 表」，包含：

- `revenue`：每月總營收
- `orders`：每月訂單數
- `aov`：平均訂單金額

提示：可以使用 `facts["order_date"].dt.to_period("M")` 建立月份欄位。


In [8]:
facts["order_month"] = facts["order_date"].dt.to_period("M")

monthly = facts.groupby("order_month", as_index=False).agg(
    revenue=("line_revenue", "sum"),
    orders=("order_id", "count"),
)
monthly["aov"] = (monthly["revenue"] / monthly["orders"]).round(2)

monthly


,order_month,revenue,orders,aov
0,2024-01,4402531.90,874,5037.22
1,2024-02,3997855.35,798,5009.84
2,2024-03,4485427.00,890,5039.81
3,2024-04,4126290.05,798,5170.79
4,2024-05,4420498.35,902,4900.77
5,2024-06,4174493.60,878,4754.55
6,2024-07,4128031.40,830,4973.53
7,2024-08,4195987.50,834,5031.16
8,2024-09,4437823.95,893,4969.57
9,2024-10,4396027.70,892,4928.28


## 9. 常見錯誤與延伸

常見錯誤：
- 忘記先確認資料是否只包含完成訂單，導致取消訂單也被算進 KPI。
- 用 `count` 時沒有想清楚粒度：目前 `facts` 是訂單層級，所以 `order_id` 的筆數就是訂單數。
- 分組日期時沒有先確認 `order_date` 是否為 datetime 型別。

延伸練習：
- 在每日 KPI 表中加入 `weekday`，比較平日與週末表現。
- 用 `pivot_table` 建立「月份 x 付款方式」的總營收表。
